# Análisis de Puntos de Venta - Consorcio de Transportes de la Bahía de Cádiz
## Paso 1: Identificación de la fuente de datos

In [1]:
organismo_fuente = "Consejería de Fomento y Movilidad (Junta de Andalucía)"
licencia_uso = "Reconocimiento 4.0 Internacional (CC BY 4.0)"
nombre_dataset = "Puntos de venta del Consorcio de Transportes de la Bahía de Cádiz"
url_dataset_web = "https://www.juntadeandalucia.es/datosabiertos/portal/dataset/consorcio-de-transporte-bahia-de-cadiz-puntos-de-venta"
url_csv_directo = "https://www.juntadeandalucia.es/datosabiertos/portal/dataset/8555d8b2-5688-4481-bf8c-733020dfc484/resource/300b6ba5-c04d-42c3-8185-687c5d5a8314/download/puntos-de-venta-consorcio-de-transporte-cadiz.csv"

print("=== FICHA DE FUENTE DE DATOS ===")
print(f"• Dataset: {nombre_dataset}")
print(f"• Organismo Público: {organismo_fuente}")
print(f"• Licencia de Uso: {licencia_uso}")
print(f"• URL del Portal: {url_dataset_web}")
print(f"• URL de Descarga Directa CSV: {url_csv_directo}")

=== FICHA DE FUENTE DE DATOS ===
• Dataset: Puntos de venta del Consorcio de Transportes de la Bahía de Cádiz
• Organismo Público: Consejería de Fomento y Movilidad (Junta de Andalucía)
• Licencia de Uso: Reconocimiento 4.0 Internacional (CC BY 4.0)
• URL del Portal: https://www.juntadeandalucia.es/datosabiertos/portal/dataset/consorcio-de-transporte-bahia-de-cadiz-puntos-de-venta
• URL de Descarga Directa CSV: https://www.juntadeandalucia.es/datosabiertos/portal/dataset/8555d8b2-5688-4481-bf8c-733020dfc484/resource/300b6ba5-c04d-42c3-8185-687c5d5a8314/download/puntos-de-venta-consorcio-de-transporte-cadiz.csv


## Paso 2: Carga de datos y exploración inicial

In [2]:
import pandas as pd

df = pd.read_csv(url_csv_directo, sep=";", encoding="latin-1", skiprows=1)

print("--- 1. PRIMERAS FILAS DEL DATAFRAME ---")
display(df.head())

print("\n--- 2. DIMENSIONES (Filas, Columnas) ---")
print(df.shape)

print("\n--- 3. TIPOS DE DATOS INICIALES ---")
print(df.dtypes)

--- 1. PRIMERAS FILAS DEL DATAFRAME ---


,TIPO,MUNICIPIO,NÚCLEO,DIRECCIÓN,LATITUD,LONGITUD,URL
0,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Urb. Espíritu Santo, L 25-26",36774299,-6371899,http://apirtod.dipucadiz.es/api/datos/puntos_v...
1,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"PÂº. Jardín Botánico, portal 2 (p-3)",36769548,-6354767,http://apirtod.dipucadiz.es/api/datos/puntos_v...
2,Comercio,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Plaza JaÃ©n, 1",36795763,-6337953,http://apirtod.dipucadiz.es/api/datos/puntos_v...
3,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Cl. Sebastián Elcano, 11","3,68E+15","-6,36E+15",http://apirtod.dipucadiz.es/api/datos/puntos_v...
4,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Cl. Santo Domingo, 90","3,68E+15","-6,35E+15",http://apirtod.dipucadiz.es/api/datos/puntos_v...



--- 2. DIMENSIONES (Filas, Columnas) ---
(247, 7)

--- 3. TIPOS DE DATOS INICIALES ---
TIPO         object
MUNICIPIO    object
NÚCLEO       object
DIRECCIÓN    object
LATITUD      object
LONGITUD     object
URL          object
dtype: object


In [3]:
print("--- 4. ANÁLISIS DE VALORES NULOS ---")
nulos_abs = df.isna().sum()
nulos_porcentual = (df.isna().mean() * 100).round(2)

df_nulos = pd.DataFrame({
    'Nulos Absolutos': nulos_abs,
    'Porcentaje (%)': nulos_porcentual
})
display(df_nulos)

--- 4. ANÁLISIS DE VALORES NULOS ---


,Nulos Absolutos,Porcentaje (%)
TIPO,2,0.81
MUNICIPIO,3,1.21
NÚCLEO,3,1.21
DIRECCIÓN,3,1.21
LATITUD,3,1.21
LONGITUD,3,1.21
URL,3,1.21


## Paso 3: Transformaciones de datos
### 3.1. Limpieza de registros y metadatos del footer

In [4]:
df = df.dropna(subset=['TIPO']).copy()
print(f"Filas restantes tras eliminar el footer: {len(df)}")

Filas restantes tras eliminar el footer: 245


### 3.2. Normalización de coordenadas geográficas

In [5]:
def normalizar_coord(serie, es_latitud=True):
    # Limpiar espacios y cambiar posibles comas por puntos
    s = serie.astype(str).str.strip().str.replace(',', '.', regex=False)
    nums = pd.to_numeric(s, errors='coerce')

    # Si viene como entero de 7 u 8 dígitos (ej. 36529780 o -6292897), dividir entre 10^6
    mask_enteros = nums.abs() > 180
    nums.loc[mask_enteros] = nums.loc[mask_enteros] / 1e6

    # Filtrar coordenadas válidas en el entorno de Cádiz para descartar errores de origen
    if es_latitud:
        nums = nums.where((nums >= 35.0) & (nums <= 38.0))
    else:
        nums = nums.where((nums >= -8.0) & (nums <= -4.0))

    return nums

df['LATITUD'] = normalizar_coord(df['LATITUD'], es_latitud=True)
df['LONGITUD'] = normalizar_coord(df['LONGITUD'], es_latitud=False)

print(f"Coordenadas válidas recuperadas: {df[['LATITUD', 'LONGITUD']].dropna().shape[0]} de {len(df)}")
display(df[['MUNICIPIO', 'DIRECCIÓN', 'LATITUD', 'LONGITUD']].head())

Coordenadas válidas recuperadas: 64 de 245


,MUNICIPIO,DIRECCIÓN,LATITUD,LONGITUD
0,Sanlucar de Barrameda,"Urb. Espíritu Santo, L 25-26",36.774299,-6.371899
1,Sanlucar de Barrameda,"PÂº. Jardín Botánico, portal 2 (p-3)",36.769548,-6.354767
2,Sanlucar de Barrameda,"Plaza JaÃ©n, 1",36.795763,-6.337953
3,Sanlucar de Barrameda,"Cl. Sebastián Elcano, 11",NaN,NaN
4,Sanlucar de Barrameda,"Cl. Santo Domingo, 90",NaN,NaN


### 3.3. Agrupación y resumen por tipo de establecimiento

In [6]:
resumen_tipos = df['TIPO'].value_counts()
print("--- RESUMEN: PUNTOS DE VENTA POR TIPO ---")
display(resumen_tipos)

--- RESUMEN: PUNTOS DE VENTA POR TIPO ---


,count
TIPO,
Estanco,170
Comercio,45
Taquilla,15
Kiosko,14
URL: https://datos.gob.es/es/catalogo/l02000011-consorcio-de-transporte-bahia-de-cadiz-puntos-de-venta1,1


## Paso 4: Visualización espacial (Mapa)

In [8]:
import plotly.express as px

df_grafica = df.dropna(subset=['LATITUD', 'LONGITUD']).copy()

fig = px.scatter_mapbox(
    df_grafica,
    lat="LATITUD",
    lon="LONGITUD",
    color="TIPO",
    hover_name="DIRECCIÓN",
    hover_data=["TIPO", "MUNICIPIO"],
    zoom=9.5,
    height=600,
    mapbox_style="open-street-map"  # 100% libre, nunca pide API Key
)

fig.update_layout(
    mapbox=dict(
        center=dict(
            lat=df_grafica["LATITUD"].mean(),
            lon=df_grafica["LONGITUD"].mean()
        )
    ),
    margin={"r": 0, "t": 0, "l": 0, "b": 0}
)

fig.show()